In [1]:
import os, glob
os.makedirs('/kaggle/working/data/athleticspose', exist_ok=True)

!wget -q -O /kaggle/working/data/athleticspose.zip \
  "https://github.com/SZucchini/AthleticsPose/releases/latest/download/data.zip"
!unzip -q /kaggle/working/data/athleticspose.zip -d /kaggle/working/data/athleticspose/
!pip install -q mediapipe


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 70.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 7.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [2]:
import mediapipe as mp
print(mp.__version__)  # should print something like 0.10.x


2026-03-10 13:41:57.835216: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773150118.135756      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773150118.224287      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773150118.944705      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773150118.944796      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773150118.944799      55 computation_placer.cc:177] computation placer alr

0.10.32


In [3]:
import os

# See top-level structure
print(os.listdir('/kaggle/working/data/athleticspose/'))

# Go one level deeper on whatever folder appears
for item in os.listdir('/kaggle/working/data/athleticspose/'):
    full_path = f'/kaggle/working/data/athleticspose/{item}'
    if os.path.isdir(full_path):
        print(f"\n{item}/")
        print(os.listdir(full_path)[:10])  # first 10 items only


['data']

data/
['LICENSES', 'LICENSE-DATASET', 'AthleticsPoseDataset']


In [4]:
ap_root = '/kaggle/working/data/athleticspose/data/AthleticsPoseDataset'

# Level 1
print("AthleticsPoseDataset/")
print(os.listdir(ap_root))

# Level 2
for item in os.listdir(ap_root):
    full = f'{ap_root}/{item}'
    if os.path.isdir(full):
        print(f"\n{item}/")
        print(os.listdir(full)[:10])


AthleticsPoseDataset/
['raw_markers_in_world', 'gt_markers3d_by_cam', 'det_markers2d_by_cam_ft', 'det_markers2d_by_cam_coco', 'camera_params', 'gt_bboxes2d_by_cam', 'gt_markers2d_by_cam']

raw_markers_in_world/
['sprint', 'sd', 'hurdle', 'running', 'racewalk', 'discus', 'shotput', 'javelin']

gt_markers3d_by_cam/
['sprint', 'sd', 'hurdle', 'running', 'racewalk', 'discus', 'shotput', 'javelin']

det_markers2d_by_cam_ft/
['sprint', 'sd', 'hurdle', 'running', 'racewalk', 'discus', 'shotput', 'javelin']

det_markers2d_by_cam_coco/
['sprint', 'sd', 'hurdle', 'running', 'racewalk', 'discus', 'shotput', 'javelin']

camera_params/
['20250216_sd.json', '20250125_shotput.json', '20250215_hurdle_bkup.json', '20250215_running.json', '20250126_sprint.json', '20250215_hurdle.json', '20250215_running_bkup.json', '20250126_running.json', '20250126_racewalk.json', '20250125_discus.json']

gt_bboxes2d_by_cam/
['sprint', 'sd', 'hurdle', 'running', 'racewalk', 'discus', 'shotput', 'javelin']

gt_markers2d

In [5]:
ap_root = '/kaggle/working/data/athleticspose/data/AthleticsPoseDataset'

# Check one action folder inside gt_markers3d_by_cam
print("gt_markers3d_by_cam/sprint/")
print(os.listdir(f'{ap_root}/gt_markers3d_by_cam/sprint')[:10])

# Check camera_params JSON to understand camera IDs
import json
cam_files = os.listdir(f'{ap_root}/camera_params')
with open(f'{ap_root}/camera_params/{cam_files[0]}') as f:
    sample_cam = json.load(f)
print("\nCamera params keys:", list(sample_cam.keys()) if isinstance(sample_cam, dict) else type(sample_cam))
print("Sample:", str(sample_cam)[:300])

# Check one action folder in gt_markers2d_by_cam
print("\ngt_markers2d_by_cam/sprint/")
sprint_files = os.listdir(f'{ap_root}/gt_markers2d_by_cam/sprint')
print(sprint_files[:10])

# Peek at one of those files
import numpy as np
sample_file = f'{ap_root}/gt_markers2d_by_cam/sprint/{sprint_files[0]}'
print("\nFile extension:", sprint_files[0].split('.')[-1])
# If it's .npy:
if sprint_files[0].endswith('.npy'):
    data = np.load(sample_file)
    print("Shape:", data.shape)


gt_markers3d_by_cam/sprint/
['S04', 'S12', 'S15', 'S23', 'S21', 'S08', 'S22', 'S06', 'S14']

Camera params keys: ['Name', 'MeasurementCreation', 'ExportTime', 'User', 'QtmVersion', 'CurrentFrame', 'TraceRange', 'Timebase', 'Cameras']
Sample: {'Name': '20250216_144839.qca', 'MeasurementCreation': '2025-02-16T14:48:39.526', 'ExportTime': '2025-02-19T00:55:39.408', 'User': 'Nagoya University Informatics', 'QtmVersion': '2023.3 (build 12577)', 'CurrentFrame': 1, 'TraceRange': {'Start': 1, 'End': 1}, 'Timebase': {'Frequency': 100.0, 'Range':

gt_markers2d_by_cam/sprint/
['S04', 'S12', 'S15', 'S23', 'S21', 'S08', 'S22', 'S06', 'S14']

File extension: S04


In [6]:
ap_root = '/kaggle/working/data/athleticspose/data/AthleticsPoseDataset'

# Go inside subject folder
print("gt_markers2d_by_cam/sprint/S04/")
s04_files = os.listdir(f'{ap_root}/gt_markers2d_by_cam/sprint/S04')
print(s04_files[:10])

# Check file type
sample = f'{ap_root}/gt_markers2d_by_cam/sprint/S04/{s04_files[0]}'
print("\nFilename:", s04_files[0])
if s04_files[0].endswith('.npy'):
    data = np.load(sample)
    print("Shape:", data.shape)
elif s04_files[0].endswith('.json'):
    with open(sample) as f: d = json.load(f)
    print("Keys:", list(d.keys()) if isinstance(d, dict) else str(d)[:200])

# Also check 3D markers same path
print("\ngt_markers3d_by_cam/sprint/S04/")
s04_3d = os.listdir(f'{ap_root}/gt_markers3d_by_cam/sprint/S04')
print(s04_3d[:10])
sample_3d = f'{ap_root}/gt_markers3d_by_cam/sprint/S04/{s04_3d[0]}'
if s04_3d[0].endswith('.npy'):
    data3d = np.load(sample_3d)
    print("3D Shape:", data3d.shape)

# Check Cameras list in camera params
with open(f'{ap_root}/camera_params/{cam_files[0]}') as f:
    cam_data = json.load(f)
cameras = cam_data.get('Cameras', {})
print("\nCameras type:", type(cameras))
print("Cameras content (truncated):", str(cameras)[:500])


gt_markers2d_by_cam/sprint/S04/
['20250216_09_5.npy', '20250216_13_1.npy', '20250216_13_0.npy', '20250216_09_3.npy', '20250126_12_6.npy', '20250126_12_7.npy', '20250126_12_1.npy', '20250216_12_3.npy', '20250216_10_7.npy', '20250216_10_3.npy']

Filename: 20250216_09_5.npy
Shape: (86, 17, 3)

gt_markers3d_by_cam/sprint/S04/
['20250216_09_5.npz', '20250126_10_7.npz', '20250216_12_4.npz', '20250126_12_2.npz', '20250216_10_2.npz', '20250126_12_0.npz', '20250126_11_6.npz', '20250126_12_6.npz', '20250216_10_7.npz', '20250126_11_5.npz']

Cameras type: <class 'list'>
Cameras content (truncated): [{'Active': True, 'Calibrated': True, 'PointCount': 983, 'AvgResidual': 0.786466, 'ExposureDelay': 0, 'Serial': 27848, 'Model': 'Miqus Video', 'ViewRotation': 0, 'FovMarker': {'Left': 0, 'Top': 0, 'Right': 1919, 'Bottom': 1087}, 'FovVideo': {'Left': 0, 'Top': 0, 'Right': 1919, 'Bottom': 1087}, 'SensorSize': {'HeightMM': 5.984, 'WidthMM': 10.56}, 'FovMarkerMax': {'Left': 0, 'Top': 0, 'Right': 1919, 'Bott

In [7]:
ap_root = '/kaggle/working/data/athleticspose/data/AthleticsPoseDataset'

# Check raw_markers_in_world structure
print("raw_markers_in_world/sprint/")
sprint_raw = os.listdir(f'{ap_root}/raw_markers_in_world/sprint')
print(sprint_raw[:5])
sample_raw = f'{ap_root}/raw_markers_in_world/sprint/{sprint_raw[0]}'
print("Extension:", sprint_raw[0].split('.')[-1])
if sprint_raw[0].endswith('.npy'):
    d = np.load(sample_raw)
    print("raw shape:", d.shape)
elif sprint_raw[0].endswith('.npz'):
    d = np.load(sample_raw)
    print("raw keys:", list(d.keys()))
    for k in d.keys(): print(f"  {k}: {d[k].shape}")

# Check gt_markers3d_by_cam NPZ
print("\ngt_markers3d_by_cam/sprint/S04/")
s04_3d = os.listdir(f'{ap_root}/gt_markers3d_by_cam/sprint/S04')
sample_3d = f'{ap_root}/gt_markers3d_by_cam/sprint/S04/{s04_3d[0]}'
d3 = np.load(sample_3d)
print("keys:", list(d3.keys()))
for k in d3.keys(): print(f"  {k}: {d3[k].shape}")


raw_markers_in_world/sprint/
['S04', 'S12', 'S15', 'S23', 'S21']
Extension: S04

gt_markers3d_by_cam/sprint/S04/
keys: ['markers_h36m', 'p2mm']
  markers_h36m: (86, 17, 3)
  p2mm: (86,)


In [17]:
import scipy.io, glob, os

penn_root  = '/kaggle/input/datasets/kaushalbora18/penn-action-dataset/Penn_Action'
labels_dir = f'{penn_root}/labels'
frames_dir = f'{penn_root}/frames'

# Verify path works
print("Labels found:", len(glob.glob(f'{labels_dir}/*.mat')))

# Inspect one .mat file fully
mat = scipy.io.loadmat(f'{labels_dir}/0001.mat')
print("\nKeys:", list(mat.keys()))
for k, v in mat.items():
    if not k.startswith('_'):
        print(f"  {k}: type={type(v)}, value={str(v)[:100]}")

Labels found: 2326

Keys: ['__header__', '__version__', '__globals__', 'action', 'pose', 'x', 'y', 'visibility', 'train', 'bbox', 'dimensions', 'nframes']
  action: type=<class 'numpy.ndarray'>, value=['baseball_pitch']
  pose: type=<class 'numpy.ndarray'>, value=['back']
  x: type=<class 'numpy.ndarray'>, value=[[136.5  135.   102.75 ... 102.   141.75  93.  ]
 [131.25 134.25 102.75 ... 102.   139.5   93.  ]
 [
  y: type=<class 'numpy.ndarray'>, value=[[ 71.25 102.75  99.   ... 255.75 324.75 321.  ]
 [ 66.75 102.    98.25 ... 255.   324.   320.25]
 [
  visibility: type=<class 'numpy.ndarray'>, value=[[1 1 0 ... 1 1 1]
 [1 1 0 ... 1 1 1]
 [1 1 0 ... 1 1 1]
 ...
 [1 0 1 ... 1 1 1]
 [1 0 1 ... 1 1 1]

  train: type=<class 'numpy.ndarray'>, value=[[-1]]
  bbox: type=<class 'numpy.ndarray'>, value=[[ 86.25  45.75 168.75 350.25]
 [ 86.25  41.25 168.   349.5 ]
 [ 86.25  41.25 168.   348.75]
 [ 86.2
  dimensions: type=<class 'numpy.ndarray'>, value=[[360 480 151]]
  nframes: type=<class 'numpy

In [18]:
import scipy.io, glob, os
import numpy as np

penn_root  = '/kaggle/input/datasets/kaushalbora18/penn-action-dataset/Penn_Action'
labels_dir = f'{penn_root}/labels'
frames_dir = f'{penn_root}/frames'

VALID_POSES   = {'front', 'side'}
VALID_ACTIONS = {'jumping_jacks', 'jump_rope', 'squat', 'pull_ups',
                 'bench_press', 'jumping', 'tennis_serve', 'baseball_pitch'}

penn_files = []  # list of mat file paths

for mat_file in sorted(glob.glob(f'{labels_dir}/*.mat')):
    mat    = scipy.io.loadmat(mat_file)
    action = str(mat['action'][0]).strip()
    pose   = str(mat['pose'][0]).strip()      # 'front', 'side', 'back'
    if action in VALID_ACTIONS and pose in VALID_POSES:
        penn_files.append(mat_file)

print(f"Penn Action kept sequences: {len(penn_files)}")


Penn Action kept sequences: 323


In [19]:
# Verify NPZ key before running full Cell 3
import numpy as np
ap_root = '/kaggle/working/data/athleticspose/data/AthleticsPoseDataset'

test = np.load(f'{ap_root}/gt_markers3d_by_cam/sprint/S04/{os.listdir(ap_root + "/gt_markers3d_by_cam/sprint/S04")[0]}')
print("Keys:", list(test.keys()))
print("markers_h36m shape:", test['markers_h36m'].shape)
# Expected: (86, 17, 3)


Keys: ['markers_h36m', 'p2mm']
markers_h36m shape: (86, 17, 3)


In [21]:
import os, glob, numpy as np, pandas as pd
from scipy.signal import savgol_filter

# ── Constants ──────────────────────────────────────────────────────────────────
ap_root      = '/kaggle/working/data/athleticspose/data/AthleticsPoseDataset'
ACTIONS      = ['sprint', 'running', 'hurdle', 'racewalk', 'sd', 'discus', 'shotput', 'javelin']
SIDE_CAM_IDS = {'1', '2', '5', '6'}
WINDOW       = 30
FEATURE_COLS = ['knee_L', 'knee_R', 'hip_L', 'hip_R', 'trunk_lean', 'asymmetry']

# ── H36M-17 joint indices ──────────────────────────────────────────────────────
# 0=Pelvis,1=RHip,2=RKnee,3=RAnkle,4=LHip,5=LKnee,6=LAnkle
# 7=Spine,8=Thorax,11=LShoulder,14=RShoulder
H36M = {'pelvis':0, 'rh':1, 'rk':2, 'ra':3,
        'lh':4,  'lk':5, 'la':6,
        'thorax':8, 'ls':11, 'rs':14}

# ── Helper functions ───────────────────────────────────────────────────────────
def angle3d(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba, bc  = a - b, c - b
    cos     = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return np.degrees(np.arccos(np.clip(cos, -1.0, 1.0)))

def features_from_3d(kpts):
    """kpts: (T, 17, 3) — H36M 3D keypoints from markers_h36m key"""
    rows = []
    for t in range(len(kpts)):
        k      = kpts[t]
        knee_L = angle3d(k[H36M['lh']], k[H36M['lk']], k[H36M['la']])
        knee_R = angle3d(k[H36M['rh']], k[H36M['rk']], k[H36M['ra']])
        hip_L  = angle3d(k[H36M['ls']], k[H36M['lh']], k[H36M['lk']])
        hip_R  = angle3d(k[H36M['rs']], k[H36M['rh']], k[H36M['rk']])
        vec    = k[H36M['thorax']] - k[H36M['pelvis']]
        trunk  = np.degrees(np.arctan2(np.abs(vec[0]), np.abs(vec[1]) + 1e-8))
        asym   = abs(knee_L - knee_R)
        rows.append([knee_L, knee_R, hip_L, hip_R, trunk, asym])
    return np.array(rows)

# ── Build ap_files list ────────────────────────────────────────────────────────
ap_files = []
for action in ACTIONS:
    pattern = f'{ap_root}/gt_markers3d_by_cam/{action}/S*/*.npz'
    for npz_file in glob.glob(pattern):
        fname  = os.path.basename(npz_file)        # e.g. 20250216_09_5.npz
        cam_id = fname.rsplit('_', 1)[-1].split('.')[0]   # '5'
        if cam_id in SIDE_CAM_IDS:
            ap_files.append((npz_file, action))

print(f"AthleticsPose files to process: {len(ap_files)}")

# ── Main extraction loop ───────────────────────────────────────────────────────
all_windows = []   # Penn Action (Cell 4) will append to this same list

skipped = 0
for npz_path, action in ap_files:
    try:
        d    = np.load(npz_path)
        kpts = d['markers_h36m']          # (T, 17, 3) confirmed
    except Exception as e:
        skipped += 1
        continue

    if kpts.ndim != 3 or kpts.shape[1] != 17 or len(kpts) < WINDOW:
        skipped += 1
        continue

    feats = features_from_3d(kpts)
    wl    = min(11, (len(feats) // 2) * 2 - 1)
    if wl < 3:
        skipped += 1
        continue

    for c in range(feats.shape[1]):
        feats[:, c] = savgol_filter(feats[:, c], window_length=wl, polyorder=3)

    for start in range(0, len(feats) - WINDOW, WINDOW // 2):
        w    = feats[start:start + WINDOW]
        flat = {'source': 'athleticspose', 'action': action}
        for fi in range(WINDOW):
            for ci, col in enumerate(FEATURE_COLS):
                flat[f'{col}_f{fi}'] = w[fi, ci]
        all_windows.append(flat)

print(f"Done. Windows extracted: {len(all_windows)} | Skipped files: {skipped}")
print("Run Cell 4 next to append Penn Action — do NOT clear all_windows")


AthleticsPose files to process: 1999
Done. Windows extracted: 13962 | Skipped files: 6
Run Cell 4 next to append Penn Action — do NOT clear all_windows


In [22]:
from scipy.signal import savgol_filter

# Penn Action joint indices
PA = {'ls':1, 'rs':2, 'lh':7, 'rh':8, 'lk':9, 'rk':10, 'la':11, 'ra':12}

def angle2d(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba, bc  = a - b, c - b
    cos     = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return np.degrees(np.arccos(np.clip(cos, -1.0, 1.0)))

def features_from_penn(mat):
    """
    x, y: (T, 13) — pre-labelled joint coords
    visibility: (T, 13) — 1=visible, 0=occluded
    """
    x   = mat['x'].astype(float)   # (T, 13)
    y   = mat['y'].astype(float)   # (T, 13)
    vis = mat['visibility']        # (T, 13)
    T   = x.shape[0]
    rows = []
    for t in range(T):
        def pt(j): return [x[t, j], y[t, j]]
        # Skip frame if key joints are occluded
        key_joints = [PA['lh'], PA['rh'], PA['lk'], PA['rk'], PA['la'], PA['ra']]
        if any(vis[t, j] == 0 for j in key_joints):
            continue
        knee_L = angle2d(pt(PA['lh']), pt(PA['lk']), pt(PA['la']))
        knee_R = angle2d(pt(PA['rh']), pt(PA['rk']), pt(PA['ra']))
        hip_L  = angle2d(pt(PA['ls']), pt(PA['lh']), pt(PA['lk']))
        hip_R  = angle2d(pt(PA['rs']), pt(PA['rh']), pt(PA['rk']))
        ms     = np.mean([pt(PA['ls']), pt(PA['rs'])], axis=0)
        mh     = np.mean([pt(PA['lh']), pt(PA['rh'])], axis=0)
        trunk  = np.degrees(np.arctan2(abs(ms[0]-mh[0]), abs(ms[1]-mh[1])+1e-8))
        asym   = abs(knee_L - knee_R)
        rows.append([knee_L, knee_R, hip_L, hip_R, trunk, asym])
    return np.array(rows) if len(rows) >= WINDOW else None

for mat_file in penn_files:
    mat    = scipy.io.loadmat(mat_file)
    action = str(mat['action'][0]).strip()
    feats  = features_from_penn(mat)
    if feats is None: continue
    wl = min(11, (len(feats) // 2) * 2 - 1)  # savgol needs odd window < data length
    if wl < 3: continue
    for c in range(feats.shape[1]):
        feats[:, c] = savgol_filter(feats[:, c], window_length=wl, polyorder=3)
    for start in range(0, len(feats) - WINDOW, WINDOW // 2):
        w    = feats[start:start+WINDOW]
        flat = {'source': 'penn_action', 'action': action}
        for fi in range(WINDOW):
            for ci, col in enumerate(FEATURE_COLS):
                flat[f'{col}_f{fi}'] = w[fi, ci]
        all_windows.append(flat)

df = pd.DataFrame(all_windows)
df.to_csv('/kaggle/working/features_raw.csv', index=False)
print(f"Total windows: {len(df)}")
print(df['source'].value_counts())


Total windows: 14349
source
athleticspose    13962
penn_action        387
Name: count, dtype: int64


In [23]:
import pandas as pd
import numpy as np

df = pd.read_csv('/kaggle/working/features_raw.csv')
FEATURE_COLS = ['knee_L', 'knee_R', 'hip_L', 'hip_R', 'trunk_lean', 'asymmetry']

# Compute per-window mean for each feature (for labeling only)
summary = pd.DataFrame()
for col in FEATURE_COLS:
    f_cols = [c for c in df.columns if c.startswith(f'{col}_f')]
    summary[col] = df[f_cols].mean(axis=1)

def composite_label(row):
    score = 0
    if row['knee_L'] < 30 or row['knee_R'] < 30:    score += 2
    elif row['knee_L'] < 45 or row['knee_R'] < 45:  score += 1
    if row['asymmetry'] > 15:                        score += 2
    elif row['asymmetry'] > 8:                       score += 1
    if row['trunk_lean'] > 20:                       score += 1
    if row['hip_L'] < 20 or row['hip_R'] < 20:      score += 1
    return 'High' if score >= 4 else ('Medium' if score >= 2 else 'Low')

df['label'] = summary.apply(composite_label, axis=1)
df.to_csv('/kaggle/working/features_labeled.csv', index=False)

dist = df['label'].value_counts()
pct  = df['label'].value_counts(normalize=True) * 100
print(dist)
print()
print(pct.round(1))

high_pct = pct.get('High', 0)
if high_pct < 10:
    print("\nHigh Risk < 10% — will need SMOTE + mirroring in Cell 6")
elif high_pct < 15:
    print("\nHigh Risk < 15% — will use class-weighted loss in LSTM")
else:
    print("\nLabel distribution acceptable")


label
Medium    10966
Low        3383
Name: count, dtype: int64

label
Medium    76.4
Low       23.6
Name: proportion, dtype: float64

⚠️  High Risk < 10% — will need SMOTE + mirroring in Cell 6


In [24]:
print(summary.describe().round(2))


         knee_L    knee_R     hip_L     hip_R  trunk_lean  asymmetry
count  14349.00  14349.00  14349.00  14349.00    14349.00   14349.00
mean     130.21    129.96    132.81    133.74       19.43      33.42
std       28.43     26.49     37.35     33.80       21.12      24.04
min       34.26     33.40     35.53     41.18        0.26       0.17
25%      115.68    118.06    115.91    109.19        3.24      11.32
50%      128.73    128.82    148.68    147.56        9.13      30.08
75%      150.62    148.20    160.84    161.37       29.94      52.47
max      178.52    179.03    179.03    178.88       87.32     107.35


In [27]:
def risk_score(row):
    score = 0
    
    # HIGH knee angle = straighter leg = ACL risk (opposite of before)
    # Danger zone: knee > 150° (nearly straight during landing/movement)
    score += (row['knee_L'] / 180) * 2
    score += (row['knee_R'] / 180) * 2
    
    # HIGH asymmetry = risk. Normalise to this dataset's range (max ~107)
    score += row['asymmetry'] / 60
    
    # HIGH trunk lean = risk
    score += row['trunk_lean'] / 45
    
    # HIGH hip angle = over-extended hip = risk
    score += (row['hip_L'] / 180)
    score += (row['hip_R'] / 180)
    
    return score

summary['risk_score'] = summary.apply(risk_score, axis=1)

p65 = summary['risk_score'].quantile(0.65)
p85 = summary['risk_score'].quantile(0.85)

print(f"Risk score stats:\n{summary['risk_score'].describe().round(3)}")
print(f"\np65: {p65:.3f} | p85: {p85:.3f}")

def assign_label(score):
    if score >= p85:   return 'High'
    elif score >= p65: return 'Medium'
    else:              return 'Low'

df = pd.read_csv('/kaggle/working/features_raw.csv')
df['label'] = summary['risk_score'].apply(assign_label)
df.to_csv('/kaggle/working/features_labeled.csv', index=False)

dist = df['label'].value_counts()
pct  = df['label'].value_counts(normalize=True) * 100
print(f"\n{dist}\n")
print(pct.round(1))


Risk score stats:
count    14349.000
mean         5.360
std          0.702
min          1.933
25%          5.197
50%          5.550
75%          5.774
max          7.374
Name: risk_score, dtype: float64

p65: 5.690 | p85: 5.888

label
Low       9327
Medium    2869
High      2153
Name: count, dtype: int64

label
Low       65.0
Medium    20.0
High      15.0
Name: proportion, dtype: float64


In [28]:
from sklearn.model_selection import train_test_split

df = pd.read_csv('/kaggle/working/features_labeled.csv')
feature_cols = [c for c in df.columns if c not in ['source', 'action', 'label']]

# ── Augmentation 1: Gaussian noise ──────────────────────────────────────────
noisy = df.copy()
noisy[feature_cols] = noisy[feature_cols] + np.random.normal(0, 1.5, noisy[feature_cols].shape)
noisy['source'] = 'aug_noise'

# ── Augmentation 2: Left-right mirror ───────────────────────────────────────
mirrored = df.copy()
for base in ['knee', 'hip']:
    l_cols = [c for c in feature_cols if f'{base}_L' in c]
    r_cols = [c for c in feature_cols if f'{base}_R' in c]
    mirrored[l_cols] = df[r_cols].values
    mirrored[r_cols] = df[l_cols].values
mirrored['source'] = 'aug_mirror'

# ── Augmentation 3: Speed perturbation (0.8x — subsample every 1.25 frames) ─
# Approximated by taking frames 0,1,2...23 and repeating last 6 to pad to 30
speed = df.copy()
for col_base in ['knee_L','knee_R','hip_L','hip_R','trunk_lean','asymmetry']:
    frame_cols = sorted([c for c in feature_cols if c.startswith(f'{col_base}_f')])
    vals = speed[frame_cols].values                    # (N, 30)
    subsampled = vals[:, :24]                          # take first 24 frames (0.8x)
    padded = np.concatenate([subsampled,               # pad last 6 by repeating f23
                             np.repeat(subsampled[:, -1:], 6, axis=1)], axis=1)
    speed[frame_cols] = padded
speed['source'] = 'aug_speed'

aug_df = pd.concat([df, noisy, mirrored, speed], ignore_index=True)
aug_df.to_csv('/kaggle/working/features_augmented.csv', index=False)

print(f"Before augmentation: {len(df)}")
print(f"After augmentation:  {len(aug_df)}")
print(f"\nLabel distribution after augmentation:")
print(aug_df['label'].value_counts())


Before augmentation: 14349
After augmentation:  57396

Label distribution after augmentation:
label
Low       37308
Medium    11476
High       8612
Name: count, dtype: int64


In [29]:
from sklearn.model_selection import train_test_split
import pandas as pd

aug_df = pd.read_csv('/kaggle/working/features_augmented.csv')

# Val and Test use ONLY original rows (not augmented)
orig_df  = aug_df[aug_df['source'].isin(['athleticspose', 'penn_action'])].copy()
aug_only = aug_df[~aug_df['source'].isin(['athleticspose', 'penn_action'])].copy()

train_base, valtest = train_test_split(orig_df, test_size=0.30,
                                       stratify=orig_df['label'], random_state=42)
val_df, test_df     = train_test_split(valtest, test_size=0.50,
                                       stratify=valtest['label'], random_state=42)

train_df = pd.concat([train_base, aug_only], ignore_index=True)

train_df.to_csv('/kaggle/working/train.csv', index=False)
val_df.to_csv('/kaggle/working/val.csv',     index=False)
test_df.to_csv('/kaggle/working/test.csv',   index=False)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"\nTrain label dist:\n{train_df['label'].value_counts()}")
print(f"\nVal label dist:\n{val_df['label'].value_counts()}")
print(f"\nTest label dist:\n{test_df['label'].value_counts()}")


Train: 53091 | Val: 2152 | Test: 2153

Train label dist:
label
Low       34510
Medium    10615
High       7966
Name: count, dtype: int64

Val label dist:
label
Low       1399
Medium     430
High       323
Name: count, dtype: int64

Test label dist:
label
Low       1399
Medium     431
High       323
Name: count, dtype: int64
